# ============================================================
# STEP 4B — INTEGRATED GRADIENTS (deep models) — TUMC
# ============================================================
# SHAP/LIME explain the TREE models (Step 4). Neural models need
# gradient-based attribution. This applies Integrated Gradients
# (Captum) to:
#   - TabNet            (engineered 134-feature input)
#   - URL-Transformer   (raw character input) — per-character attribution
#
# IG computes the integral of gradients along a straight-line path
# from a baseline to the input, satisfying the completeness axiom:
#   sum of attributions = f(input) − f(baseline).
#
# Output: step4b_ig_feature_attr.csv (TabNet feature attributions),
#         ig_plots/*.png (incl. per-character URL heatmap)
#
# Requires: torch, captum, pytorch-tabnet
#   !pip install captum pytorch-tabnet -q
# ============================================================

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 42

DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
for cand in ["features_full_final_inductive.csv","features_full_final.csv"]:
    FEAT_FILE = os.path.join(DATA_DIR, cand)
    if os.path.exists(FEAT_FILE): break
PLOTS = os.path.join(DATA_DIR, "ig_plots"); os.makedirs(PLOTS, exist_ok=True)

TASK = "binary"
LEAKY = ["cluster_malicious_ratio","campaign_token_score","is_tr_domain","is_https"]

try:
    import torch
    from captum.attr import IntegratedGradients
    HAS_TORCH = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    print("pip install captum torch -q"); raise SystemExit

print("="*70)
print(f"STEP 4B — INTEGRATED GRADIENTS (deep models, {TASK})")
print(f"  Device: {DEVICE}")
print("="*70)

df = pd.read_csv(FEAT_FILE, low_memory=False)
META = {"url","source","tld","label","label_enc","class_final",
        "class_enc","fold","reg_domain"}
FEATURES = [c for c in df.columns if c not in META and c not in LEAKY]
target = "label_enc" if TASK=="binary" else "class_enc"
n_classes = df[target].nunique()

# ════════════════════════════════════════════════════════════
# PART A — IG on TabNet (engineered features)
# ════════════════════════════════════════════════════════════
print(f"\n[A] Integrated Gradients on TabNet ...")
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    from sklearn.preprocessing import StandardScaler
    from sklearn.impute import SimpleImputer

    tr = df[df["fold"]!=0]; te = df[df["fold"]==0]
    imp = SimpleImputer(strategy="median"); sc = StandardScaler()
    X_tr = sc.fit_transform(imp.fit_transform(tr[FEATURES].values))
    X_te = sc.transform(imp.transform(te[FEATURES].values))
    y_tr = tr[target].values.astype(int); y_te = te[target].values.astype(int)

    nval = int(0.1*len(X_tr)); perm = np.random.RandomState(SEED).permutation(len(X_tr))
    vi, fi = perm[:nval], perm[nval:]
    clf = TabNetClassifier(n_d=32, n_a=32, n_steps=5, gamma=1.5,
        seed=SEED, verbose=0, device_name=DEVICE)
    clf.fit(X_tr[fi], y_tr[fi], eval_set=[(X_tr[vi], y_tr[vi])],
            max_epochs=60, patience=12, batch_size=4096, virtual_batch_size=512)

    # Captum IG on the TabNet network
    net = clf.network.to(DEVICE).eval()
    ig = IntegratedGradients(lambda x: net(x)[0])  # TabNet returns (logits, M_loss)
    n_attr = min(2000, len(X_te))
    ai = np.random.RandomState(SEED).choice(len(X_te), n_attr, replace=False)
    X_attr = torch.tensor(X_te[ai], dtype=torch.float32, device=DEVICE)
    target_cls = torch.tensor(y_te[ai], dtype=torch.long, device=DEVICE)
    baseline = torch.zeros_like(X_attr)
    attr = ig.attribute(X_attr, baseline, target=target_cls, n_steps=50)
    ig_imp = np.abs(attr.cpu().detach().numpy()).mean(0)
    ig_s = pd.Series(ig_imp, index=FEATURES).sort_values(ascending=False)

    print("    TabNet IG top-10 features:")
    for f,v in ig_s.head(10).items(): print(f"      {f:<26s}: {v:.4f}")

    ig_df = pd.DataFrame({"feature":ig_s.index, "ig_attribution":ig_s.values})
    ig_df.to_csv(os.path.join(DATA_DIR,"step4b_ig_feature_attr.csv"), index=False)

    # Compare with SHAP top features if available
    try:
        with open(os.path.join(DATA_DIR,"shap_top_features.json")) as fh:
            shap_top = set(json.load(fh)["top15"])
        ig_top = set(ig_s.head(15).index)
        ov = shap_top & ig_top
        print(f"    SHAP(trees) ∩ IG(TabNet) top-15: {len(ov)} shared → {sorted(ov)[:6]}")
    except FileNotFoundError:
        pass

    plt.figure(figsize=(10,7))
    ig_s.head(20).iloc[::-1].plot(kind="barh", color="#7B5BA6")
    plt.xlabel("Mean |Integrated Gradients attribution|")
    plt.title(f"TabNet — Integrated Gradients ({TASK})", fontweight="bold")
    plt.tight_layout(); plt.savefig(f"{PLOTS}/ig_tabnet_{TASK}.png", dpi=150); plt.close()
    print(f"    Saved {PLOTS}/ig_tabnet_{TASK}.png")
except Exception as e:
    print(f"    TabNet IG skipped: {type(e).__name__}: {e}")

# ════════════════════════════════════════════════════════════
# PART B — IG on URL-Transformer (per-character attribution)
# ════════════════════════════════════════════════════════════
print(f"\n[B] Integrated Gradients on URL-Transformer (char-level) ...")
MODEL_PT = os.path.join(DATA_DIR, "url_transformer_best.pt")
if not os.path.exists(MODEL_PT):
    print(f"    {MODEL_PT} not found — train Step 7 first. Skipping.")
else:
    try:
        import math, torch.nn as nn
        CHARS = ("abcdefghijklmnopqrstuvwxyz0123456789"
                 ".-_/:?=&%@+~#!$*()[]{}|\\;,<>\"' çğıöşüâî")
        char2idx = {ch:i+2 for i,ch in enumerate(CHARS)}; PAD=0; UNK=1
        VOCAB=len(char2idx)+2; MAXLEN=200; D=64
        def enc(u):
            u=str(u).lower()[:MAXLEN]; ids=[char2idx.get(ch,UNK) for ch in u]
            return ids+[PAD]*(MAXLEN-len(ids)), u

        class PE(nn.Module):
            def __init__(s,d,ml=MAXLEN):
                super().__init__(); pe=torch.zeros(ml,d)
                pos=torch.arange(0,ml).unsqueeze(1).float()
                dv=torch.exp(torch.arange(0,d,2).float()*(-math.log(10000)/d))
                pe[:,0::2]=torch.sin(pos*dv); pe[:,1::2]=torch.cos(pos*dv)
                s.register_buffer("pe",pe.unsqueeze(0))
            def forward(s,x): return x+s.pe[:,:x.size(1)]
        class URLT(nn.Module):
            def __init__(s,v,nc):
                super().__init__()
                s.embed=nn.Embedding(v,D,padding_idx=PAD); s.pos=PE(D)
                el=nn.TransformerEncoderLayer(D,4,128,0.1,activation="gelu",batch_first=True)
                s.enc=nn.TransformerEncoder(el,3)
                s.head=nn.Sequential(nn.Linear(D*2,D),nn.GELU(),nn.Linear(D,nc))
                s.emb_cache=None
            def embed_in(s,x):
                e=s.embed(x)*math.sqrt(D); return s.pos(e)
            def forward_from_emb(s,emb,mask):
                h=s.enc(emb,src_key_padding_mask=mask)
                hm=h.masked_fill(mask.unsqueeze(-1),0).sum(1)/(~mask).sum(1,keepdim=True).clamp(min=1)
                hx=h.masked_fill(mask.unsqueeze(-1),-1e9).max(1).values
                return s.head(torch.cat([hm,hx],1))
            def forward(s,x):
                mask=(x==PAD); return s.forward_from_emb(s.embed_in(x),mask)

        net=URLT(VOCAB, 2 if TASK=="binary" else n_classes).to(DEVICE)
        net.load_state_dict(torch.load(MODEL_PT, map_location=DEVICE)); net.eval()

        # IG over the EMBEDDING (standard for token models)
        examples = df[df["class_final"]!="benign"]["url"].head(3).tolist() + \
                   df[df["class_final"]=="benign"]["url"].head(1).tolist()
        fig, axes = plt.subplots(len(examples),1, figsize=(14, 2.2*len(examples)))
        if len(examples)==1: axes=[axes]
        for ax,url in zip(axes, examples):
            ids,u = enc(url)
            x=torch.tensor([ids],device=DEVICE)
            mask=(x==PAD)
            emb=net.embed_in(x).detach().requires_grad_(True)
            ig=IntegratedGradients(lambda e: net.forward_from_emb(e,mask))
            base=torch.zeros_like(emb)
            att=ig.attribute(emb, base, target=1, n_steps=50)
            char_attr=att.sum(dim=2).squeeze(0).detach().cpu().numpy()[:len(u)]
            char_attr=char_attr/ (np.abs(char_attr).max()+1e-9)
            ax.imshow(char_attr.reshape(1,-1), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
            ax.set_xticks(range(len(u))); ax.set_xticklabels(list(u), fontsize=7)
            ax.set_yticks([]); ax.set_title(f"{url[:70]}", fontsize=8, loc="left")
        plt.suptitle(f"URL-Transformer — Per-Character Integrated Gradients ({TASK})\n"
                     "red = pushes toward malicious, blue = toward benign", fontweight="bold")
        plt.tight_layout(); plt.savefig(f"{PLOTS}/ig_url_chars_{TASK}.png", dpi=150, bbox_inches="tight"); plt.close()
        print(f"    Saved {PLOTS}/ig_url_chars_{TASK}.png  (per-character heatmap)")
    except Exception as e:
        print(f"    URL-Transformer IG skipped: {type(e).__name__}: {e}")

print(f"""
{'='*70}
STEP 4B (DEEP XAI) COMPLETE
{'='*70}
  Integrated Gradients applied to TabNet + URL-Transformer.
  TabNet feature attributions: step4b_ig_feature_attr.csv
  Per-character URL heatmap: ig_plots/ig_url_chars_{TASK}.png

  Full XAI story now spans:
    SHAP + LIME + Permutation  → tree models (Step 4)
    Integrated Gradients       → deep models (Step 4B)
  Cross-method agreement = method-independent explanations.
{'='*70}
""")

STEP 4B — INTEGRATED GRADIENTS (deep models, binary)
  Device: cuda

[A] Integrated Gradients on TabNet ...

Early stopping occurred at epoch 50 with best_epoch = 38 and best_val_0_auc = 0.99944
    TabNet IG top-10 features:
      domain_ngram_cluster      : 2.5050
      num_slashes               : 1.6715
      path_depth                : 0.8185
      path_entropy              : 0.6773
      num_dots                  : 0.4323
      tld_token_cooccur         : 0.2965
      alpha_ratio               : 0.2326
      has_www                   : 0.2019
      subdomain_reuse_count     : 0.1978
      is_suspicious_tld         : 0.1923
    SHAP(trees) ∩ IG(TabNet) top-15: 6 shared → ['domain_ngram_cluster', 'is_suspicious_tld', 'num_dots', 'num_slashes', 'path_depth', 'tld_token_cooccur']
    Saved /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/ig_plots/ig_tabnet_binary.png

[B] Integrated Gradients on URL-Transformer (char-level) ...
    /content/drive/MyDrive/Colab Notebooks